*Merge Bevölkerung und Beschuldigungen*

In [1]:
import pandas as pd

In [2]:
bev = pd.read_parquet("bevoelkerung.parquet")
besch = pd.read_parquet("beschuldigte.parquet")

In [3]:
# ständige Wohnbevölkerung, nur Total / Schweiz / Ausland
pop = bev[
    (bev["Bevölkerungstyp"] == "Ständige Wohnbevölkerung")
    & (bev["Geburtsstaat"] == "Geburtsstaat - Total")
    & (bev["Staatsangehörigkeit"].isin(["Staatsangehörigkeit - Total", "Schweiz", "Ausland"]))
].copy()

# gleiche Bezeichnung wie bei den Beschuldigten
pop["Aufenthaltsgruppe"] = pop["Staatsangehörigkeit"].astype(str).map({
    "Staatsangehörigkeit - Total": "Total alle Beschuldigte",
    "Schweiz": "Total Schweizer Beschuldigte",
    "Ausland": "Total Beschuldigte aus der ständigen ausländischen Wohnbevölkerung",
})

pop = pop[["Jahr", "Kanton", "Geschlecht", "Aufenthaltsgruppe", "DATA"]].rename(columns={"DATA": "Einwohner"})
pop.head()

,Jahr,Kanton,Geschlecht,Aufenthaltsgruppe,Einwohner
0,2010,Schweiz,Geschlecht - Total,Total alle Beschuldigte,7870134.0
1,2010,Schweiz,Mann,Total alle Beschuldigte,3877426.0
2,2010,Schweiz,Frau,Total alle Beschuldigte,3992708.0
598,2010,Schweiz,Geschlecht - Total,Total Schweizer Beschuldigte,6103857.0
599,2010,Schweiz,Mann,Total Schweizer Beschuldigte,2939447.0


In [4]:
# keine Bevölkerung nach Alter -> nur Altersklasse Total
# Bevölkerungsdaten erst ab 2010
besch = besch[
    (besch["Altersklasse"] == "Altersklasse - Total")
    & (besch["Jahr"] >= 2010)
]
besch = besch.drop(columns="Altersklasse").rename(columns={"DATA": "Beschuldigte"})
besch.shape

(1730160, 6)

*Merge der beiden datensätze*

In [5]:
df = besch.merge(pop, on=["Jahr", "Kanton", "Geschlecht", "Aufenthaltsgruppe"], how="left")
df["Rate_100k"] = df["Beschuldigte"] / df["Einwohner"] * 100_000

print(df.shape)
df.head()

(1730160, 8)


,Straftat,Kanton,Aufenthaltsgruppe,Geschlecht,Jahr,Beschuldigte,Einwohner,Rate_100k
0,Straftat - Total,Schweiz,Total alle Beschuldigte,Geschlecht - Total,2010,79229.0,7870134.0,1006.704587
1,Straftat - Total,Schweiz,Total alle Beschuldigte,Geschlecht - Total,2011,77637.0,7954662.0,975.993700
2,Straftat - Total,Schweiz,Total alle Beschuldigte,Geschlecht - Total,2012,81682.0,8039060.0,1016.064067
3,Straftat - Total,Schweiz,Total alle Beschuldigte,Geschlecht - Total,2013,81236.0,8139631.0,998.030500
4,Straftat - Total,Schweiz,Total alle Beschuldigte,Geschlecht - Total,2014,79069.0,8237666.0,959.847122


*kontrollen*

In [6]:
# gleich viele Zeilen wie vorher?
print(len(df) == len(besch))

# Asylbevölkerung und übrige Ausländer haben keine Einwohnerzahl
df.groupby("Aufenthaltsgruppe")["Einwohner"].apply(lambda x: x.isna().mean())

True


Aufenthaltsgruppe
Total Beschuldigte aus der Asylbevölkerung                            1.0
Total Beschuldigte aus der ständigen ausländischen Wohnbevölkerung    0.0
Total Schweizer Beschuldigte                                          0.0
Total alle Beschuldigte                                               0.0
Total übrige ausländische Beschuldigte                                1.0
Name: Einwohner, dtype: float64

In [7]:
# ganze Schweiz, alle Straftaten
df[
    (df["Straftat"] == "Straftat - Total")
    & (df["Kanton"] == "Schweiz")
    & (df["Geschlecht"] == "Geschlecht - Total")
    & (df["Aufenthaltsgruppe"] == "Total alle Beschuldigte")
][["Jahr", "Beschuldigte", "Einwohner", "Rate_100k"]]

,Jahr,Beschuldigte,Einwohner,Rate_100k
0,2010,79229.0,7870134.0,1006.704587
1,2011,77637.0,7954662.0,975.993700
2,2012,81682.0,8039060.0,1016.064067
3,2013,81236.0,8139631.0,998.030500
4,2014,79069.0,8237666.0,959.847122
5,2015,77062.0,8327126.0,925.433337
6,2016,78240.0,8419550.0,929.265816
7,2017,78184.0,8484130.0,921.532320
8,2018,80345.0,8544527.0,940.309510
9,2019,81709.0,8606033.0,949.438609


In [8]:
# Neuchâtel, Diebstahl
df[
    (df["Straftat"] == "Diebstahl (Art. 139)")
    & (df["Kanton"] == "Neuchâtel")
    & (df["Geschlecht"] == "Geschlecht - Total")
    & (df["Aufenthaltsgruppe"] == "Total alle Beschuldigte")
][["Jahr", "Beschuldigte", "Einwohner", "Rate_100k"]]

,Jahr,Beschuldigte,Einwohner,Rate_100k
154800,2010,188.0,172085.0,109.248337
154801,2011,174.0,173183.0,100.471755
154802,2012,207.0,174554.0,118.587944
154803,2013,211.0,176402.0,119.613156
154804,2014,186.0,177327.0,104.890964
154805,2015,180.0,178107.0,101.062844
154806,2016,163.0,178567.0,91.282264
154807,2017,170.0,177964.0,95.524938
154808,2018,151.0,176850.0,85.383093
154809,2019,178.0,176496.0,100.852144


In [9]:
df.to_parquet("merged.parquet", index=False)